In [1]:
import sys
import os
from pathlib import Path

# Walk upward until we find the folder that contains "src"
p = Path.cwd().resolve()
while p != p.parent and not (p / "src").exists():
    p = p.parent

if not (p / "src").exists():
    raise RuntimeError("Could not find repo root (folder containing 'src').")

if str(p) not in sys.path:
    sys.path.insert(0, str(p))

print("Repo root set to:", p)


Repo root set to: /home/iauger/projects/dsci632-project


In [2]:
import os, subprocess
import sys, platform

print("python:", sys.executable)
print("platform:", platform.platform())
print("cwd:", os.getcwd())
print("home:", str(Path.home()))

assert "/home/" in str(Path.home()), "Not running in WSL home"
assert "mnt/c" not in os.getcwd().lower(), "CWD is on Windows mount; use ~/projects/... instead"
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))
print("which java:", subprocess.check_output(["which","java"]).decode().strip())

python: /home/iauger/projects/dsci632-project/.venv/bin/python
platform: Linux-5.15.167.4-microsoft-standard-WSL2-x86_64-with-glibc2.39
cwd: /home/iauger/projects/dsci632-project/notebooks
home: /home/iauger
JAVA_HOME: /usr/lib/jvm/java-17-openjdk-amd64
which java: /usr/lib/jvm/java-17-openjdk-amd64/bin/java


In [ ]:
from src.config import load_settings 

s = load_settings()

ENV: local
RAW recipes: /home/iauger/projects/dsci632-project/data/raw/RAW_recipes.csv
RAW interactions: /home/iauger/projects/dsci632-project/data/raw/RAW_interactions.csv
PROCESSED: /home/iauger/projects/dsci632-project/data/processed
BRONZE: /home/iauger/projects/dsci632-project/data/processed/bronze
SILVER: /home/iauger/projects/dsci632-project/data/processed/silver
GOLD: /home/iauger/projects/dsci632-project/data/processed/gold
RUN_ID: 20260227_011552
FEATURES DATASET /home/iauger/projects/dsci632-project/data/processed/features/runs/20260227_011552/dataset.parquet


In [4]:
from src.spark.session import get_spark
    
spark = get_spark(app_name="testing-spark-session", debug=True)

print("spark.version:", spark.version)
print("spark.driver.host:", spark.conf.get("spark.driver.host", "NOT SET"))
print("spark.local.ip:", spark.conf.get("spark.local.ip", "NOT SET"))

your 131072x1 screen size is bogus. expect trouble
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/27 01:19:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


spark.version: 3.5.1
spark.driver.host: 127.0.0.1
spark.local.ip: 127.0.0.1


In [ ]:
import shutil
from pathlib import Path

targets = [
    Path(s.bronze_recipes_path),
    Path(s.bronze_interactions_path),
    Path(s.silver_recipes_path),
    Path(s.silver_interactions_path),
    Path(s.gold_ve_path),
    Path(s.gold_cf_path),
]

for t in targets:
    if t.exists():
        shutil.rmtree(t)
        print("deleted:", t)
    else:
        print("missing (ok):", t)

In [ ]:
recipes_raw = (
    spark.read
    .option("header", "true")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("mode", "PERMISSIVE")
    .option("enforceSchema", "false")
    .csv(s.raw_recipes_path)
)

interactions_raw = (
    spark.read
    .option("header", "true")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("mode", "PERMISSIVE")
    .option("enforceSchema", "false")
    .csv(s.raw_interactions_path)
)

recipes_raw.write.mode("overwrite").parquet(s.bronze_recipes_path)
interactions_raw.write.mode("overwrite").parquet(s.bronze_interactions_path)

In [ ]:
recipes_b = spark.read.parquet(s.bronze_recipes_path)

# id should be numeric-ish and definitely not long sentences
recipes_b.select("id", "nutrition").show(10, truncate=False)

# sanity: find “bad id” rows (contains spaces or punctuation typical of text)
from pyspark.sql import functions as F
bad = recipes_b.filter(F.col("id").rlike(r"[A-Za-z]|\s"))
bad.select("id", "nutrition").show(20, truncate=False)
print("bad id count:", bad.count())

In [ ]:
from src.spark.ingestion.spark.transforms.recipes_spark import clean_recipes     
from src.spark.ingestion.spark.transforms.interactions_spark import clean_interactions 

recipes_b = spark.read.parquet(s.bronze_recipes_path)
interactions_b = spark.read.parquet(s.bronze_interactions_path)

recipes_s = clean_recipes(recipes_b)
interactions_s = clean_interactions(interactions_b)

recipes_s.write.mode("overwrite").parquet(s.silver_recipes_path)
interactions_s.write.mode("overwrite").parquet(s.silver_interactions_path)

print("silver written:")
print(" -", s.silver_recipes_path)
print(" -", s.silver_interactions_path)

In [ ]:
interactions_s = spark.read.parquet(s.silver_interactions_path)
interactions_s.printSchema()

In [ ]:
interactions_s.show(10, truncate=False)

In [ ]:
interactions_s.printSchema()
interactions_s.groupBy("rating").count().orderBy("rating").show()

In [ ]:
recipes_s.printSchema()

In [ ]:
from pyspark.sql import functions as F

recipes_s = spark.read.parquet(s.silver_recipes_path)
interactions_s = spark.read.parquet(s.silver_interactions_path)

print("recipes_s:", recipes_s.count(), "cols:", len(recipes_s.columns))
print("interactions_s:", interactions_s.count(), "cols:", len(interactions_s.columns))

# check null rates on cleaned fields
recipes_s.select(
    F.sum(F.col("ingredients_clean").isNull().cast("int")).alias("null_ingredients_clean"),
    F.sum(F.col("steps_clean").isNull().cast("int")).alias("null_steps_clean"),
    F.sum(F.col("tags_clean").isNull().cast("int")).alias("null_tags_clean"),
    F.sum(F.col("description_clean").isNull().cast("int")).alias("null_description_clean"),
).show()

interactions_s.select(
    F.sum(F.col("review_clean").isNull().cast("int")).alias("null_review_clean"),
    F.sum(F.col("rating").isNull().cast("int")).alias("null_rating"),
).show()

# confirm rating 0 removed
interactions_s.groupBy("rating").count().orderBy("rating").show(10)

In [ ]:
print(s.gold_reviews_path)

In [ ]:
from src.spark.ingestion.spark.transforms.merge_spark import build_gold_recipes, build_gold_reviews
from pyspark import StorageLevel

recipes_s = spark.read.parquet(s.silver_recipes_path)
interactions_s = spark.read.parquet(s.silver_interactions_path)

gold_reviews = build_gold_reviews(interactions_s)
gold_recipes = build_gold_recipes(recipes_s)

gold_reviews.write.mode("overwrite").parquet(s.gold_reviews_path)
gold_recipes.write.mode("overwrite").parquet(s.gold_recipe_path)

In [ ]:
from src.spark.ingestion.spark.transforms.merge_spark import build_gold_ve

gold_ve = build_gold_ve(gold_recipes, gold_reviews)

gold_ve.coalesce(32).write.mode("overwrite").parquet(s.gold_ve_path)
print("Wrote gold_ve")

In [ ]:
gold_reviews.schema

In [ ]:
# Row counts and column count review
print("DF SHAPE")
print("VE:", gold_ve.count(), "cols:", len(gold_ve.columns))

# Like distribution
print(40 * "-")
print("\nLIKE DISTRIBUTION")
gold_ve.groupBy("liked").count().show()

tot = gold_ve.count()
pos = gold_ve.filter(F.col("liked") == 1).count()
pct_pos = pos / tot * 100
print(f"Positive class (liked=1) count: {pos} / {tot} ({pct_pos:.2f}%)")

# Rating distribution
print(40 * "-")
print("\nRATING DISTRIBUTION")
gold_ve.groupBy("rating").count().orderBy("rating").show()

# Top reviewed recipes
print(40 * "-")
print("\nTOP 10 RECIPES BY REVIEW COUNT")
top10 = (
    gold_ve.groupBy("recipe_id", "name")
         .agg(
             F.count("*").alias("n_reviews"),
             F.round(F.avg("rating"), 3).alias("avg_rating"),
             F.sum(F.col("liked").cast("int")).alias("n_liked"),
         )
         .orderBy(F.desc("n_reviews"), F.desc("avg_rating"))
         .limit(10)
)
top10.show(truncate=60)


# Review length distribution (chars)
print(40 * "-")
print("\nREVIEW LENGTH (chars) SUMMARY")
review_len = gold_ve.withColumn("review_len", F.length(F.coalesce(F.col("review_clean"), F.lit(""))))

review_len.select(
    F.min("review_len").alias("min_len"),
    F.round(F.avg("review_len"), 2).alias("avg_len"),
    F.max("review_len").alias("max_len"),
).show()

# Percentiles (more informative than min/max)
print(40 * "-")
print("\nREVIEW LENGTH (chars) PERCENTILES")
review_len.selectExpr("percentile_approx(review_len, array(0.5, 0.9, 0.95, 0.99)) as p").show(truncate=False)

# Null counts (VE required cols)
print(40 * "-")
print("\nNULL COUNTS (VE REQUIRED)")
required_ve = [
    "user_id","recipe_id","name","rating","liked","date","review_clean",
    "ingredients_clean","steps_clean","tags_clean","description_clean",
    "minutes","n_steps","n_ingredients",
    "calories","fat","sugar","sodium","protein","saturated_fat","carbs",
]
existing_required_ve = [c for c in required_ve if c in gold_ve.columns]

gold_ve.select(
    *[F.sum(F.col(c).isNull().cast("int")).alias(f"null_{c}") for c in existing_required_ve]
).show(truncate=False)

In [ ]:
from src.spark.labeling.zero_shot import ZeroShotConfig, prepare_zero_shot_input_from_gold, run_zero_shot_incremental_with_checkpoints


# Updated Config for the overnight marathon
cfg_overnight = ZeroShotConfig(
    taxonomy_version="v1",
    min_tokens=15,
    sample_n=20000, # Set this to your desired total target
    batch_size=4,   # Dropped slightly to be kinder to CPU thermal throttling overnight
    sample_seed=42, # Keep seed same to ensure the same 'random' order
)

# 1. Prepare input (this generates the full 'target' list)
inp = prepare_zero_shot_input_from_gold(
    spark=spark,
    gold_reviews_path=s.gold_reviews_path,  
    cfg=cfg_overnight,
    processed_dir=s.processed_dir,
)

# 2. Run with the incremental helper
print("Starting incremental zero-shot run...")
out = run_zero_shot_incremental_with_checkpoints(inp, cfg_overnight, s.processed_dir)

In [ ]:
import pandas as pd
from pathlib import Path

out_path = "path/to/your/processed/labeling/zero_shot/zs_output.parquet" # Update this

if Path(out_path).exists():
    df_check = pd.read_parquet(out_path)
    print(f"--- Overnight Run Summary ---")
    print(f"Total samples labeled: {len(df_check)}")
    print(f"Unique review keys:    {df_check['review_key'].nunique()}")
    print(f"Avg labels per review: {df_check['zs_num_labels'].mean():.2f}")
    print(f"Fallback usage rate:   {(df_check['zs_used_fallback'].sum() / len(df_check)) * 100:.1f}%")
    
    # Check for empty labels (potential threshold issues)
    no_labels = df_check[df_check['zs_num_labels'] == 0].shape[0]
    print(f"Reviews with 0 labels: {no_labels}")
else:
    print("Output file not found. Check your logs for errors!")

In [ ]:
from src.spark.labeling.zero_shot import attach_zero_shot_labels_to_gold
labeled = attach_zero_shot_labels_to_gold(spark, s.gold_reviews_path, s.processed_dir, cfg)
print("labeled parquet:", labeled)

In [ ]:
df_labeled = spark.read.parquet(labeled)
print("Labeled DF schema:")
df_labeled.printSchema()
print("Labeled DF sample:")
df_labeled.show(10, truncate=False)

In [ ]:
import pandas as pd
import json
from collections import Counter, defaultdict

from src.spark.labeling.taxonomy import get_taxonomy

def run_zs_diagnostics_pandas(zs_out_path, taxonomy_version="v1", top_k_tags=20):
    pdf = pd.read_parquet(zs_out_path)
    tax = get_taxonomy(taxonomy_version)
    pol_map = {tid: t.polarity for tid, t in tax.items()}

    def normalize_labels(x):
        if x is None:
            return []
        if isinstance(x, float) and pd.isna(x):
            return []
        if isinstance(x, str):
            return [x] if x else []
        if hasattr(x, "tolist"):
            x = x.tolist()
        if isinstance(x, (list, tuple, set)):
            return [t for t in x if t is not None and not (isinstance(t, float) and pd.isna(t))]
        return []

    n = len(pdf)
    print("n_reviews:", n)
    print("avg_labels_per_review:", round(pdf["zs_num_labels"].mean(), 3))
    print("median_labels_per_review:", pdf["zs_num_labels"].median())
    print("fallback_rate:", round(pdf["zs_used_fallback"].mean(), 4))
    print("avg_max_score:", round(pdf["zs_max_score"].mean(), 4))

    # per-tag prevalence
    tag_ctr = Counter()
    for labels in pdf["zs_labels"]:
        tag_ctr.update(normalize_labels(labels))
    print(f"\nTop {top_k_tags} tags (% of reviews):")
    for tid, cnt in tag_ctr.most_common(top_k_tags):
        print(f"{tid:28s} {cnt/max(n, 1):.3f}  ({pol_map.get(tid,'?')})")

    # polarity distribution over ASSIGNED labels
    pol_ctr = Counter()
    for labels in pdf["zs_labels"]:
        for tid in normalize_labels(labels):
            pol_ctr[pol_map.get(tid, "unknown")] += 1
    total_assigned = sum(pol_ctr.values()) or 1
    print("\nPolarity distribution over assigned labels:")
    for pol in ["positive", "negative", "neutral", "unknown"]:
        print(f"{pol:10s} {pol_ctr[pol]/total_assigned:.3f}")

    # % reviews with >=1 tag of each polarity
    def has_pol(labels, pol):
        labels = normalize_labels(labels)
        return any(pol_map.get(t) == pol for t in labels)

    print("\nReview-level polarity presence:")
    print("pct_reviews_with_pos:", round(pdf["zs_labels"].apply(lambda x: has_pol(x, "positive")).mean(), 3))
    print("pct_reviews_with_neg:", round(pdf["zs_labels"].apply(lambda x: has_pol(x, "negative")).mean(), 3))
    print("pct_reviews_with_neu:", round(pdf["zs_labels"].apply(lambda x: has_pol(x, "neutral")).mean(), 3))

    # score means by polarity (all tag scores, not just assigned)
    pol_score_sum = defaultdict(float)
    pol_score_n = defaultdict(int)
    for s in pdf["zs_scores_json"]:
        if isinstance(s, dict):
            scores = s
        elif isinstance(s, str) and s.strip():
            scores = json.loads(s)
        else:
            scores = {}
        for tid, sc in scores.items():
            pol = pol_map.get(tid, "unknown")
            pol_score_sum[pol] += float(sc)
            pol_score_n[pol] += 1

    print("\nAverage raw score by polarity (all tag scores):")
    for pol in ["positive", "negative", "neutral", "unknown"]:
        if pol_score_n[pol]:
            print(f"{pol:10s} {pol_score_sum[pol]/pol_score_n[pol]:.4f}")

# Usage:
# run_zs_diagnostics_pandas("/home/iauger/.../zs_output.parquet", taxonomy_version="v1", top_k_tags=20)

run_zs_diagnostics_pandas(labeled, taxonomy_version=cfg.taxonomy_version, top_k_tags=20)